In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from lightgbm import LGBMClassifier
import pandas as pd


df = pd.read_csv("../DATA/PROCESS/save_dataset3.csv")

TARGET = "target"

X = df.drop(columns=[TARGET])
y = df[TARGET]
import numpy as np

# datetime 처리
if "datetime" in X.columns:
    X["datetime"] = pd.to_datetime(X["datetime"], errors="coerce")
    X["datetime_year"] = X["datetime"].dt.year
    X["datetime_month"] = X["datetime"].dt.month
    X["datetime_day"] = X["datetime"].dt.day
    X["datetime_hour"] = X["datetime"].dt.hour
    X = X.drop(columns=["datetime"])

# 문자열 컬럼 처리
cat_cols = X.select_dtypes(include=["object"]).columns

for col in cat_cols:
    X[col] = X[col].astype("category").cat.codes

# 결측치 처리
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(-1)

# target 타입 처리
y = y.astype(int)

print(X.dtypes.value_counts())
print("남은 object 컬럼:", X.select_dtypes(include=["object"]).columns.tolist())

import re

def clean_colname(col):
    col = str(col)
    col = re.sub(r'[^A-Za-z0-9_]+', '_', col)
    col = col.strip("_")
    
    if col == "":
        col = "col"
    
    return col

X.columns = [clean_colname(c) for c in X.columns]

# 중복 컬럼명 방지
counts = {}
new_cols = []

for col in X.columns:
    if col not in counts:
        counts[col] = 0
        new_cols.append(col)
    else:
        counts[col] += 1
        new_cols.append(f"{col}_{counts[col]}")

X.columns = new_cols

print("중복 컬럼 수:", X.columns.duplicated().sum())
print("컬럼 수:", len(X.columns))

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)


base_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)

base_model.fit(X_train, y_train)

base_pred = base_model.predict_proba(X_test)[:, 1]
base_auc = roc_auc_score(y_test, base_pred)

print(f"\n[BASE MODEL] ROC-AUC: {base_auc:.4f}")


feature_groups = {
    "time_feature": [
        "hour_sin", "hour_cos",
        "weekday_sin", "weekday_cos",
        "month_sin", "month_cos"
    ],

    "subway_feature": [
        "subway_cnt_300m",
        "subway_cnt_500m",
        "subway_cnt_1000m",
        "nearest_subway_dist_m",
        "subway_line_entropy_1000m"
    ],

    "bus_feature": [
        "bus_cnt_300m",
        "bus_cnt_500m",
        "nearest_bus_dist_m"
    ],

    "population_feature": [
        "local_pop",
        "tourist_pop"
    ],

    "commercial_feature": [
        "restaurant_cnt",
        "cafe_cnt",
        "hotel_cnt"
    ]
}

results = []

for group_name, cols in feature_groups.items():

    # 존재하는 컬럼만 제거
    remove_cols = [c for c in cols if c in X.columns]

    X_ablation = X.drop(columns=remove_cols)

    X_train, X_test, y_train, y_test = train_test_split(
        X_ablation, y,
        test_size=0.2,
        shuffle=False
    )

    model = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        random_state=42
    )

    model.fit(X_train, y_train)

    pred = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, pred)

    diff = base_auc - auc

    results.append({
        "removed_feature_group": group_name,
        "roc_auc": auc,
        "performance_drop": diff
    })


result_df = pd.DataFrame(results)

result_df = result_df.sort_values(
    by="performance_drop",
    ascending=False
)

print("\n===== Ablation Test Result =====")
print(result_df)

/var/folders/g8/gwqmqng10_g3h7m8r02yq1qc0000gn/T/ipykernel_83140/2486417705.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X["datetime_day"] = X["datetime"].dt.day
/var/folders/g8/gwqmqng10_g3h7m8r02yq1qc0000gn/T/ipykernel_83140/2486417705.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X["datetime_hour"] = X["datetime"].dt.hour
/var/folders/g8/gwqmqng10_g3h7m8r02yq1qc0000gn/T/ipykernel_83140/2486417705.py:32: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is

int64      64
float64    33
int32       4
int8        2
Name: count, dtype: int64
남은 object 컬럼: []
중복 컬럼 수: 0
컬럼 수: 103
[LightGBM] [Info] Number of positive: 1480624, number of negative: 1746500
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.308142 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3274
[LightGBM] [Info] Number of data points in the train set: 3227124, number of used features: 101
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.458806 -> initscore=-0.165150
[LightGBM] [Info] Start training from score -0.165150

[BASE MODEL] ROC-AUC: 0.8343
[LightGBM] [Info] Number of positive: 1480624, number of negative: 1746500
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.363510 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_